In [92]:
# GenAI & Machine Learning Bootcamp 2025 - Daily Challenge : Building a GAN-Based AI Text Detector - 24 JUILLET 2025 -
# Full Time 2025 - PSTB GenAI Introduction to Generative AI Building a GAN-Based AI Text Detector Building a GAN-Based AI Text Detector

# 🛠️ What you will create
    # A GAN-based model that detects AI-generated text using embeddings from a BERT model.
    # A training pipeline that leverages a discriminator and generator network.
    # A model that improves based on AUC scores for stability in training.
    # A final submission file with predictions on the test dataset.


# Preliminaire : importation des libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import string

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

In [93]:
# 1. Download the Dataset
    # Upload the Kaggle API key.
    # Move the key to the correct directory and set permissions, you may accept the rules of the competitions in Rulesor in Participate.
    # Download and unzip the dataset or Download manually from Kaggle

from google.colab import files

# Ouvre une fenêtre pour importer un fichier depuis ton ordinateur
uploaded = files.upload()

Saving sample_submission.csv to sample_submission (1).csv
Saving test_essays.csv to test_essays (1).csv
Saving train_essays.csv to train_essays (1).csv
Saving train_prompts.csv to train_prompts (1).csv


In [94]:
# Configuration du device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device: {device}")

Utilisation du device: cpu


In [95]:
# Définition des chemins des fichiers basés sur votre structure
TRAIN_PATH = "train_essays.csv"
TEST_PATH = "test_essays.csv"
PROMPT_PATH = "train_prompts.csv"

# Chargement des données selon votre structure
src_train = pd.read_csv(TRAIN_PATH)  # train_essays.csv
src_prompt = pd.read_csv(PROMPT_PATH)  # train_prompts.csv
src_sub = pd.read_csv(TEST_PATH)  # test_essays.csv pour les prédictions

In [96]:
# Chargement également du fichier de soumission exemple
sample_submission_df = pd.read_csv("sample_submission.csv")

print("Forme des données d'entraînement:", src_train.shape)
print("Forme des données de test:", src_sub.shape)
print("Colonnes d'entraînement:", src_train.columns.tolist())
print("Colonnes de test:", src_sub.columns.tolist())
print("Colonnes des prompts:", src_prompt.columns.tolist())

# Affichage des premières lignes pour comprendre la structure
print("\nPremières lignes de train_essays:")
print(src_train.head(2))
print("\nPremières lignes de test_essays:")
print(src_sub.head(2))
print("\nPremières lignes de train_prompts:")
print(src_prompt.head(2))

Forme des données d'entraînement: (1378, 4)
Forme des données de test: (3, 3)
Colonnes d'entraînement: ['id', 'prompt_id', 'text', 'generated']
Colonnes de test: ['id', 'prompt_id', 'text']
Colonnes des prompts: ['prompt_id', 'prompt_name', 'instructions', 'source_text']

Premières lignes de train_essays:
         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   

   generated  
0          0  
1          0  

Premières lignes de test_essays:
         id  prompt_id          text
0  0000aaaa          2  Aaa bbb ccc.
1  1111bbbb          3  Bbb ccc ddd.

Premières lignes de train_prompts:
   prompt_id                       prompt_name  \
0          0                   Car-free cities   
1          1  Does the electoral college work?   

                                        instructions  \
0  Write an explanatory essa

In [97]:
# Model preparation
tokenizer_save_path = "bert-base-uncased"  # Modèle BERT standard
model_save_path = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(tokenizer_save_path)
pretrained_model = BertForSequenceClassification.from_pretrained(model_save_path)
embedding_model = pretrained_model.bert  # Extraction de la partie BERT pour les embeddings

print("Modèles BERT chargés avec succès")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modèles BERT chargés avec succès


In [98]:
# Parameter definition
train_batch_size = 16     # Réduit pour éviter les problèmes de mémoire avec BERT
test_batch_size = 32      # Taille des batches de test
lr = 0.0002              # Taux d'apprentissage standard pour GAN
beta1 = 0.5              # Paramètre beta1 pour l'optimiseur Adam
nz = 100                 # Dimensions du vecteur latent
num_epochs = 3           # Nombre d'époques (réduit pour les tests)
num_hidden_layers = 6    # Nombre de couches cachées dans BERT
train_ratio = 0.8        # Ratio pour la division train/validation

print(f"Hyperparamètres définis - Batch size: {train_batch_size}, Learning rate: {lr}, Epochs: {num_epochs}")

Hyperparamètres définis - Batch size: 16, Learning rate: 0.0002, Epochs: 3


In [99]:
# Data Preparation
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


In [101]:
# Calcul des tailles des ensembles
all_num = len(src_train)                    # Nombre total d'échantillons
train_num = int(all_num * train_ratio)      # Nombre d'échantillons d'entraînement
test_num = all_num - train_num              # Nombre d'échantillons de validation

print(f"Division des données - Total: {all_num}, Train: {train_num}, Validation: {test_num}")

Division des données - Total: 1378, Train: 1102, Validation: 276


In [102]:
# Création des ensembles d'entraînement et de validation
train_set = src_train.iloc[:train_num].reset_index(drop=True)
test_set = pd.concat([
    src_train.iloc[train_num:].reset_index(drop=True)
]).reset_index(drop=True)

# IMPORTANT: Vous devez vérifier les noms exacts des colonnes dans vos fichiers
# Les colonnes courantes dans ce type de dataset sont généralement :
# - 'text' ou 'essay' pour le texte de l'essai
# - 'label' ou 'generated' pour indiquer si c'est généré par IA (1) ou humain (0)
# - 'id' pour l'identifiant unique

# Création des datasets PyTorch
# ATTENTION: Remplacez 'text' et 'label' par les vrais noms de colonnes de votre dataset
try:
    train_dataset = GANDAIGDataset(
        texts=train_set['text'].tolist(),  # Colonne contenant le texte
        labels=train_set['label'].tolist()  # Colonne contenant les labels (0/1)
    )

    test_dataset = GANDAIGDataset(
        texts=test_set['text'].tolist(),
        labels=test_set['label'].tolist()
    )
except KeyError as e:
    print(f"ERREUR: Colonne manquante {e}")
    print("Colonnes disponibles dans train_essays:", src_train.columns.tolist())
    print("Veuillez vérifier les noms des colonnes et les modifier dans le code")
    # Colonnes alternatives possibles :
    # - 'essay', 'content', 'full_text' pour le texte
    # - 'generated', 'is_ai', 'target' pour les labels

ERREUR: Colonne manquante 'label'
Colonnes disponibles dans train_essays: ['id', 'prompt_id', 'text', 'generated']
Veuillez vérifier les noms des colonnes et les modifier dans le code


# Définition des chemins des fichiers basés sur votre structure
TRAIN_PATH = "train_essays.csv"
TEST_PATH = "test_essays.csv"
PROMPT_PATH = "train_prompts.csv"

# Chargement des données selon votre structure
src_train = pd.read_csv(TRAIN_PATH)  # train_essays.csv
src_prompt = pd.read_csv(PROMPT_PATH)  # train_prompts.csv
src_sub = pd.read_csv(TEST_PATH)  # test_essays.csv pour les prédictions


Forme des données d'entraînement: (1378, 4)
Forme des données de test: (3, 3)
Colonnes d'entraînement: ['id', 'prompt_id', 'text', 'generated']
Colonnes de test: ['id', 'prompt_id', 'text']
Colonnes des prompts: ['prompt_id', 'prompt_name', 'instructions', 'source_text']

Premières lignes de train_essays:
         id  prompt_id                                               text  \
0  0059830c          0  Cars. Cars have been around since they became ...   
1  005db917          0  Transportation is a large necessity in most co...   

   generated  
0          0  
1          0  

Premières lignes de test_essays:
         id  prompt_id          text
0  0000aaaa          2  Aaa bbb ccc.
1  1111bbbb          3  Bbb ccc ddd.

Premières lignes de train_prompts:
   prompt_id                       prompt_name  \
0          0                   Car-free cities   
1          1  Does the electoral college work?   

                                        instructions  \
0  Write an explanatory essay to inform fellow ci...   
1  Write a letter to your state senator in which ...   

                                         source_text  
0  # In German Suburb, Life Goes On Without Cars ...  
1  # What Is the Electoral College? by the Office...  

In [90]:
# 2. Charger les fichiers de données
  # 2.1 Read the training and test datasets using pandas.
train_essays_df = pd.read_csv("train_essays.csv")
test_essays_df = pd.read_csv("test_essays.csv")
train_prompts_df = pd.read_csv("train_prompts.csv")
sample_submission_df = pd.read_csv("sample_submission.csv")
  # 2.2 Display basic statistics and structure of the dataset.
from IPython.display import display
display(train_essays_df.head(5))

  # Associer les datasets aux variables utilisées
src_train = train_essays_df
src_prompt = train_prompts_df
src_sub = sample_submission_df

,id,prompt_id,text,generated
0,0059830c,0,Cars. Cars have been around since they became ...,0
1,005db917,0,Transportation is a large necessity in most co...,0
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0
3,00940276,0,How often do you ride in a car? Do you drive a...,0
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0


In [52]:
display(test_essays_df.head(10))

,id,prompt_id,text
0,0000aaaa,2,Aaa bbb ccc.
1,1111bbbb,3,Bbb ccc ddd.
2,2222cccc,4,CCC ddd eee.


In [54]:
display(train_prompts_df.head(10))

,prompt_id,prompt_name,instructions,source_text
0,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,1,Does the electoral college work?,Write a letter to your state senator in which ...,# What Is the Electoral College? by the Office...


In [55]:
display(sample_submission_df.head(10))

,id,generated
0,0000aaaa,0.1
1,1111bbbb,0.9
2,2222cccc,0.4


In [16]:
# Afficher les statistiques globales
print("\n🔹 Dimensions :")

print(f"Train_essays shape: {train_essays_df.shape}")
print(f"Test_essays shape : {test_essays_df.shape}")
print(f"Train_prompts shape: {train_prompts_df.shape}")
print(f"Sample_submission shape : {sample_submission_df.shape}")


🔹 Dimensions :
Train_essays shape: (1378, 4)
Test_essays shape : (3, 3)
Train_prompts shape: (2, 4)
Sample_submission shape : (3, 2)


In [25]:
print(train_essays_df.describe())

         prompt_id    generated
count  1378.000000  1378.000000
mean      0.486212     0.002177
std       0.499991     0.046625
min       0.000000     0.000000
25%       0.000000     0.000000
50%       0.000000     0.000000
75%       1.000000     0.000000
max       1.000000     1.000000


In [26]:
print(test_essays_df.describe())

       prompt_id
count        3.0
mean         3.0
std          1.0
min          2.0
25%          2.5
50%          3.0
75%          3.5
max          4.0


In [27]:
print(train_prompts_df.describe())

       prompt_id
count   2.000000
mean    0.500000
std     0.707107
min     0.000000
25%     0.250000
50%     0.500000
75%     0.750000
max     1.000000


In [28]:
print(sample_submission_df.describe())

       generated
count   3.000000
mean    0.466667
std     0.404145
min     0.100000
25%     0.250000
50%     0.400000
75%     0.650000
max     0.900000


In [21]:
print(train_essays_df.dtypes)

id           object
prompt_id     int64
text         object
generated     int64
dtype: object


In [22]:
print(test_essays_df.dtypes)

id           object
prompt_id     int64
text         object
dtype: object


In [23]:
print(train_prompts_df.dtypes)

prompt_id        int64
prompt_name     object
instructions    object
source_text     object
dtype: object


In [24]:
print(sample_submission_df.dtypes)

id            object
generated    float64
dtype: object


In [20]:
# vérifier les colonnes utiles
print(train_essays_df.columns.tolist())
print(test_essays_df.columns.tolist())
print(train_prompts_df.columns.tolist())
print(sample_submission_df.columns.tolist())

['id', 'prompt_id', 'text', 'generated']
['id', 'prompt_id', 'text']
['prompt_id', 'prompt_name', 'instructions', 'source_text']
['id', 'generated']


In [68]:
# Détection des valeurs nulles / manquantes
      # Nombre de valeurs nulles par colonne
train_prompts_df.isnull().sum()

,0
prompt_id,0
prompt_name,0
instructions,0
source_text,0


In [69]:
      # Lignes contenant au moins une valeur nulle
train_prompts_df[train_essays_df.isnull().any(axis=1)].head()

/tmp/ipython-input-69-2408586000.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_prompts_df[train_essays_df.isnull().any(axis=1)].head()


,prompt_id,prompt_name,instructions,source_text


In [70]:
      # Valeurs uniques par colonne
train_prompts_df.nunique()

,0
prompt_id,2
prompt_name,2
instructions,2
source_text,2


In [71]:
      # Détection des doublons
train_prompts_df.duplicated().sum()                      # Nombre de lignes dupliquées
train_prompts_df[train_essays_df.duplicated()]           # Affiche les doublons

/tmp/ipython-input-71-3650618090.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_prompts_df[train_essays_df.duplicated()]           # Affiche les doublons


,prompt_id,prompt_name,instructions,source_text


In [72]:
      # Détection des valeurs "non renseignées" non nulles
import numpy as np

custom_na_values = ["n/a", "na", "--", "-", "?", "none", "unknown", ""]
train_prompts_df.apply(lambda col: col.isin(custom_na_values).sum())

,0
prompt_id,0
prompt_name,0
instructions,0
source_text,0


In [103]:
# 3. Prepare the Model
    # 3.1 Importer les modules nécessaires
        # Load the BERT tokenizer and pre-trained model for sequence classification : bert-base-uncased
from transformers import BertTokenizer, BertModel
import torch

tokenizer_save_path = "./bert_tokenizer"
model_save_path = "./bert_model"

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
pretrained_model = BertForSequenceClassification.from_pretrained("bert-base-uncased")
embedding_model = pretrained_model.bert  # Extraction du bloc BERT sans la tête de classification


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
        # Extract embeddings from the BERT model to use in the GAN framework.

# Exemple de phrase
text = "Artificial intelligence is transforming the world."

# Tokenisation
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Passage dans BERT
with torch.no_grad():
    outputs = bert_model(**inputs)

# Récupérer les embeddings du token [CLS] (résumé de la séquence)
cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape : [batch_size, hidden_size]

print("Embedding shape:", cls_embedding.shape)

Embedding shape: torch.Size([1, 768])


In [73]:
#4. Set Hyperparameters
    # Define batch sizes, learning rates, latent vector dimensions, and training epochs.

# Données et entraînement
BATCH_SIZE = 64
EPOCHS = 50

# GAN
LATENT_DIM = 100  # taille du vecteur aléatoire z (entrée du générateur)
LEARNING_RATE_G = 2e-4  # learning rate du générateur
LEARNING_RATE_D = 2e-4  # learning rate du discriminateur

# Embeddings BERT
EMBEDDING_DIM = 768  # pour bert-base-uncased

# Optimisation
BETAS = (0.5, 0.999)  # paramètres pour Adam

# Device
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [104]:
print(src_train.columns)

Index(['id', 'prompt_id', 'text', 'generated'], dtype='object')


In [105]:
# 5. Prepare the Data for Training
    # Create a PyTorch dataset class for handling text data.
    # Split the data into training and testing sets.
    # Use DataLoader to load batches efficiently.

    # Objectifs :
        # Créer une classe Dataset personnalisée
        # Séparer les données en train/test
        # Utiliser des DataLoaders efficaces pour gérer les batches

class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# Taille totale du dataset
all_num = len(src_train)

# Répartition train / test
train_num = int(0.8 * all_num)  # 80% pour l'entraînement
test_num = all_num - train_num  # 20% pour le test

# Séparation manuelle du dataset
train_set = src_train.iloc[:train_num]
test_set = pd.concat([
    src_train.iloc[train_num:],
]).reset_index(drop=True)

# Préparation des objets Dataset pour PyTorch
train_dataset = GANDAIGDataset(
    texts=train_set["text"].tolist(),   # ou "text" selon le nom exact de la colonne
    labels=train_set["generated"].tolist()   # adapte le nom selon ton CSV
)

test_dataset = GANDAIGDataset(
    texts=test_set["text"].tolist(),
    labels=test_set["generated"].tolist()
)

# DataLoaders PyTorch
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)


In [117]:
from transformers import BertConfig
import torch.nn as nn

#6. Define the Generator Model
    # Build a neural network that generates text embeddings using ConvTranspose1D layers.
    # Incorporate a BERT encoder in the generator.

# Configuration BERT avec nombre de couches ajusté
config = BertConfig(num_hidden_layers=num_hidden_layers)

class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Projeter le vecteur latent (nz) vers une forme compatible avec le CNN
        self.fc = nn.Linear(input_dim, 256 * 8)  # 8 pour une "longueur séquentielle" fictive

        # Déconvolutions pour générer des embeddings de style séquence
        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(in_channels=256, out_channels=128, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.ConvTranspose1d(in_channels=128, out_channels=64, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv1d(in_channels=64, out_channels=config.hidden_size, kernel_size=1),
            nn.Tanh()
        )

        # Encodeur BERT (sans tête de classification)
        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        # 1. Passer le bruit aléatoire par la couche fully connected
        x = self.fc(x)  # (batch_size, 256*8)
        x = x.view(-1, 256, 8)  # (batch_size, channels, sequence_length)

        # 2. Déconvolutions pour obtenir une "fausse" séquence d'embeddings
        x = self.conv_net(x)  # (batch_size, hidden_size, seq_len)

        # 3. Transposer pour correspondre au format attendu par BERT (batch, seq_len, hidden)
        x = x.transpose(1, 2)

        # 4. Passage dans l’encodeur BERT with attention_mask
        batch_size, seq_len, _ = x.shape
        attention_mask = torch.ones((batch_size, 1, 1, seq_len), dtype=torch.float32).to(x.device)
        encoded_output = self.bert_encoder(
            hidden_states=x,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True
        )

        return encoded_output.last_hidden_state  # or pooled_output depending on your goal

In [4]:
#7. Define the Discriminator Model
    # Extract and modify layers from a pre-trained BERT model.
    # Implement a pooling mechanism for text classification.
    # Construct a classification head using fully connected layers.

import torch
import torch.nn as nn
from transformers import BertModel
from transformers.models.bert.modeling_bert import BertEncoder # Import BertEncoder from the correct location

# Pooling personnalisé : moyenne sur les embeddings BERT d'une séquence
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: [batch_size, seq_len, hidden_size]
        sum_hidden = hidden_states.sum(dim=1)  # [batch_size, hidden_size]
        # The following lines for sum_mask and division seem incorrect for a simple sum pooler.
        # A sum pooler just sums along the sequence length dimension.
        # Removing the incorrect normalization.
        return sum_hidden


# Discriminator based on BERT (encoder + binary classifier)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        # Use a partial copy of BERT (e.g., the first 6 layers)
        # Ensure 'config' is defined or passed as an argument if not globally available
        try:
            config
        except NameError:
            from transformers import BertConfig
            config = BertConfig(num_hidden_layers=6) # Define config if not already defined

        self.bert_encoder = BertEncoder(config)
        # Assuming pretrained_model is defined and loaded elsewhere
        try:
            pretrained_model
            self.bert_encoder.layer = nn.ModuleList([
                layer for layer in pretrained_model.bert.encoder.layer[:6]
            ])
        except NameError:
             print("Warning: pretrained_model not found. Initializing BertEncoder with default weights.")


        # Output pooling of embeddings (simple sum)
        self.pooler = SumBertPooler()

        # Binary classifier (fake / real) with 2 layers
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)  # scalar output (logit)
        )

    def forward(self, input):
        # input: simulated sequence embeddings [batch, seq_len, hidden]
        # Create attention mask for the input
        batch_size, seq_len, _ = input.shape
        attention_mask = torch.ones((batch_size, 1, 1, seq_len), dtype=torch.float32).to(input.device)

        out = self.bert_encoder(input, attention_mask=attention_mask).last_hidden_state
        out = self.pooler(out)               # [batch_size, hidden_size]
        out = self.classifier(out)           # [batch_size, 1]
        return torch.sigmoid(out).view(-1)   # output: probability ∈ [0,1]

In [7]:
# Training
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

# 1. AUC Evaluation

def eval_auc(model):
    model.eval()
    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            encodings = tokenizer(batch[0], padding=True, truncation=True, return_tensors="pt").to(device)
            input_ids = encodings["input_ids"]
            token_type_ids = encodings["token_type_ids"]
            attention_mask = encodings["attention_mask"]

            embeded = embedding_model(
                input_ids=input_ids,
                token_type_ids=token_type_ids,
                attention_mask=attention_mask
            ).last_hidden_state

            label = batch[1].float().to(device)
            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc

# 2. Model state tracking

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info

# 3. Prepare BERT Embeddings

def preparation_embedding(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    input_ids = encodings['input_ids']
    token_type_ids = encodings['token_type_ids']
    attention_mask = encodings['attention_mask']
    embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
    return embeded.last_hidden_state

# 4. GAN Step

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.zero_grad()
    batch_size = real_data.size(0)
    output = netD(real_data)
    errD_real = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    noise = torch.randn(batch_size, nz, device=device)
    fake_embed = netG(noise)
    label.fill_(1)
    output = netD(fake_embed.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()
    errD = errD_real + errD_fake
    optimizerD.step()

    netG.zero_grad()
    label.fill_(0)
    output = netD(fake_embed)
    errG = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs, i, len(train_loader), errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

# 5. Updated Discriminator with attention_mask
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input):
        batch_size, seq_len, _ = input.shape
        attention_mask = torch.ones((batch_size, 1, 1, seq_len), dtype=torch.float32).to(input.device)
        out = self.bert_encoder(hidden_states=input, attention_mask=attention_mask).last_hidden_state
        out = self.pooler(out)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

In [10]:
# 6. Model init
netG = Generator(nz).to(device)
netD = Discriminator(config).to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# 7. Training loop
model_infos = []
for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        with torch.no_grad():
            embeded = preparation_embedding(data[0])

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch, i=i)

    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete!')

NameError: name 'Generator' is not defined

In [38]:
#5

    # Step 2 : Séparer les données en train/test
from sklearn.model_selection import train_test_split

# Merge sur prompt_id pour enrichir les essais avec leur prompt
train_df = train_essays_df.merge(train_prompts_df, on="prompt_id", how="left")
test_df = test_essays_df.merge(train_prompts_df, on="prompt_id", how="left")

from sklearn.model_selection import train_test_split

# Texte + cible
texts = train_df["text"].tolist()
labels = train_df["generated"].tolist()  # Remplace par la vraie colonne cible si différente

# Split 80% / 20%
texts_train, texts_val, labels_train, labels_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [76]:
    # Step 3 : Use DataLoader to load batches efficiently;
        # Importer les modules PyTorch nécessaires

import torch
from torch.utils.data import Dataset, DataLoader

        # Définir une classe Dataset pour les essais (texte + label)
class EssayDataset(Dataset):
  def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

  def __len__(self):
        return len(self.texts)

  def __getitem__(self, idx):
        # Tokeniser le texte
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Sortie de type dict
        item = {
            'input_ids': encoded["input_ids"].squeeze(0),
            'attention_mask': encoded["attention_mask"].squeeze(0)
        }

        # Ajouter les labels si fournis (pour entraînement supervisé)
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item

            # Créer les datasets à partir des splits
                # Vérification de la bonne instanciation de mon tokenizer
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

                # Datasets
train_dataset = EssayDataset(texts_train, labels_train, tokenizer)
val_dataset = EssayDataset(texts_val, labels_val, tokenizer)

          # Créer les DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

          # Visualiser un batch pour vérifier
batch = next(iter(train_loader))
print("📦 Input IDs shape:", batch["input_ids"].shape)
print("📦 Attention mask shape:", batch["attention_mask"].shape)
print("📦 Labels shape:", batch["label"].shape)

📦 Input IDs shape: torch.Size([16, 128])
📦 Attention mask shape: torch.Size([16, 128])
📦 Labels shape: torch.Size([16])


In [77]:
# 6. Define the Generator Model
    # Build a neural network that generates text embeddings using ConvTranspose1D layers.
    # Incorporate a BERT encoder in the generator.

import torch
import torch.nn as nn
from transformers import BertModel

class TextEmbeddingGenerator(nn.Module):
    def __init__(self, latent_dim=100, seq_len=32, hidden_dim=768):
        super(TextEmbeddingGenerator, self).__init__()
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim

        # Générateur (de bruit latent → embeddings séquence)
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256 * (seq_len // 4)),  # projette puis reshape
            nn.ReLU(),
            nn.Unflatten(1, (256, seq_len // 4)),         # shape: [B, 256, seq_len//4]

            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=2, padding=1),  # x2
            nn.ReLU(),

            nn.ConvTranspose1d(128, hidden_dim, kernel_size=4, stride=2, padding=1),  # x2 → [B, 768, seq_len]
            nn.Tanh()
        )

        # Encodeur BERT (optionnel pour guidance ou intégration)
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.bert.requires_grad_(False)  # Ne pas entraîner BERT dans le générateur

    def forward(self, z):
        """
        z: bruit latent, shape [batch_size, latent_dim]
        Output: fake embeddings [batch_size, seq_len, hidden_dim]
        """
        x = self.net(z)                      # [B, 768, seq_len]
        x = x.permute(0, 2, 1)               # → [B, seq_len, 768]

        # Optionnel : passer dans BERT pour guidance ou cohérence
        # attention_mask = torch.ones((x.size(0), self.seq_len), dtype=torch.long).to(x.device)
        # outputs = self.bert(inputs_embeds=x, attention_mask=attention_mask)
        # return outputs.last_hidden_state

        return x  # embeddings générés simulant ceux de BERT

In [16]:
# 7. Define the Discriminator Model
      # Extract and modify layers from a pre-trained BERT model.
      # Implement a pooling mechanism for text classification.
      # Construct a classification head using fully connected layers.

import torch
import torch.nn as nn
from transformers import BertModel, BertConfig
from transformers.models.bert.modeling_bert import BertEncoder # Import BertEncoder from the correct location


# Pooling mechanism: mean pooling
class SumBertPooler(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)  # [batch_size, hidden_size]
        sum_mask = sum_hidden.sum(1).unsqueeze(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask
        return mean_embeddings


# Discriminator definition
class Discriminator(nn.Module):
    def __init__(self, config):
        super().__init__()

        # BERT encoder config and layers
        self.bert_encoder = BertEncoder(config)
        # Assuming pretrained_model is defined and loaded elsewhere
        try:
            pretrained_model
            self.bert_encoder.layer = nn.ModuleList([
                layer for layer in pretrained_model.bert.encoder.layer[:6]
            ])
        except NameError:
             print("Warning: pretrained_model not found. Initializing BertEncoder with default weights.")


        # Pooling and classification head
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input):
        batch_size, seq_len, _ = input.shape

        # Simulate attention_mask of 1s
        attention_mask = torch.ones((batch_size, 1, 1, seq_len), dtype=torch.float32).to(input.device)

        out = self.bert_encoder(hidden_states=input, attention_mask=attention_mask).last_hidden_state
        out = self.pooler(out)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

In [30]:
# 8. Train the Model
      #       # Implement a GAN training loop.
      # Train the generator to produce embeddings that fool the discriminator.
      # Train the discriminator to differentiate between real and generated embeddings.
      # Evaluate the model using AUC scores to monitor training stability.

# Training
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from transformers.models.bert.modeling_bert import BertEncoder

# 1. AUC Evaluation

def eval_auc(model):
    model.eval()
    predictions = []
    actuals = []
    with torch.no_grad():
        for batch in test_loader:
            encodings = tokenizer(batch[0], padding=True, truncation=True, return_tensors="pt").to(device)
            input_ids = encodings["input_ids"]
            token_type_ids = encodings["token_type_ids"]
            attention_mask = encodings["attention_mask"]

            embeded = embedding_model(
                input_ids=input_ids,
                token_type_ids=token_type_ids,
                attention_mask=attention_mask
            ).last_hidden_state

            label = batch[1].float().to(device)
            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    auc = roc_auc_score(actuals, predictions)
    print("AUC:", auc)
    return auc

# 2. Model state tracking

def get_model_info_dict(model, epoch, auc_score):
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info

# 3. Prepare BERT Embeddings

def preparation_embedding(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    input_ids = encodings['input_ids']
    token_type_ids = encodings['token_type_ids']
    attention_mask = encodings['attention_mask']
    embeded = embedding_model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
    return embeded.last_hidden_state

# 4. GAN Step

def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    netD.zero_grad()
    batch_size = real_data.size(0)
    output = netD(real_data)
    errD_real = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    noise = torch.randn(batch_size, 100, device=device)  # nz = 100
    fake_embed = netG(noise)
    label.fill_(1)
    output = netD(fake_embed.detach())
    errD_fake = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()
    errD = errD_real + errD_fake
    optimizerD.step()

    netG.zero_grad()
    label.fill_(0)
    output = netD(fake_embed)
    errG = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs, i, len(train_loader), errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

# 5. Updated Discriminator with attention_mask
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert_model = BertModel.from_pretrained("bert-base-uncased")
        self.bert_model.encoder.layer = nn.ModuleList([
            layer for layer in self.bert_model.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(config.hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, input):
        batch_size, seq_len, _ = input.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.float32).to(input.device)

        # Important: use inputs_embeds instead of input_ids
        out = self.bert_model(inputs_embeds=input, attention_mask=attention_mask).last_hidden_state
        out = self.pooler(out)
        out = self.classifier(out)
        return torch.sigmoid(out).view(-1)

In [31]:
# 6. Define Generator
class Generator(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        self.seq_len = 35  # aligné avec les embeddings réels
        self.fc = nn.Linear(input_dim, 256 * self.seq_len)

        self.conv_net = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(128, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(64, config.hidden_size, kernel_size=3, stride=1, padding=1)
        )

    def forward(self, x):
        x = self.fc(x)  # (B, 256*seq_len)
        x = x.view(x.size(0), 256, self.seq_len)  # (B, 256, seq_len)
        x = self.conv_net(x)  # (B, hidden_size, seq_len)
        x = x.transpose(1, 2)  # (B, seq_len, hidden_size)
        return x

# 7. Model init
nz = 100
netG = Generator(nz, config).to(device)
netD = Discriminator(config).to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# 8. Training loop
model_infos = []
for epoch in range(num_epochs):
    for i, data in enumerate(train_loader, 0):
        with torch.no_grad():
            embeded = preparation_embedding(data[0])

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch, i=i)

    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Train complete!')

NameError: name 'config' is not defined

In [47]:
import os

print(os.listdir())  # liste les fichiers dans le répertoire courant

['.config', 'test_essays.csv', 'train_prompts.csv', 'sample_submission.csv', 'train_essays.csv', 'sample_data']


In [32]:
# 9. Perform Inference
      # Load the best-performing discriminator model based on AUC scores.
      # Process test data through the model to generate predictions.

  # 9. Perform Inference
print("\n Loading best discriminator model and running inference...")

# 1. Identifier le meilleur modèle basé sur AUC
best_model_info = max(model_infos, key=lambda x: x["auc_score"])
best_discriminator = Discriminator().to(device)
best_discriminator.load_state_dict(best_model_info["model_state_dict"])
best_discriminator.eval()

# 2. Prédictions sur les données de test
predictions = []
ids = []

with torch.no_grad():
    for batch in test_loader:
        input_texts = batch[0]
        encodings = tokenizer(input_texts, padding=True, truncation=True, return_tensors="pt").to(device)
        input_ids = encodings["input_ids"]
        token_type_ids = encodings["token_type_ids"]
        attention_mask = encodings["attention_mask"]

        embeded = embedding_model(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        outputs = best_discriminator(embeded)
        predictions.extend(outputs.cpu().numpy())
        ids.extend(batch[2]) if len(batch) > 2 else None  # optionnel si ID inclus dans le dataset

# 3. Affichage partiel
for i, pred in enumerate(predictions[:10]):
    print(f"Test sample {i + 1}: Score = {pred:.4f}")



 Loading best discriminator model and running inference...


NameError: name 'model_infos' is not defined